# The Neural Tangent Kernel

| Difficulty | ████████░░ 8/10 |
| :--- | :--- |
| Prerequisites | 11.02 |
| Time | ~85 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/11_Functional_Analysis/03_neural_tangent_kernel.ipynb)

---

## 🎯 Learning Objective

Derive the Neural Tangent Kernel and understand the lazy-training regime versus feature learning.

---

## 📐 Theory

`11.02` showed kernel methods reduce an infinite-dimensional optimization problem to finite
linear algebra on a Gram matrix. This notebook uncovers a startling fact: in the infinite-width
limit, training a neural network with gradient descent *is* kernel regression with one specific
kernel — the **Neural Tangent Kernel** — turning the black box of deep learning into something
this module already knows how to analyze exactly.

### Linearizing the network around its initialization

For a network $f_\theta$ with parameters $\theta$, a first-order Taylor expansion around the
initial parameters $\theta_0$ gives $f_\theta(\mathbf{x}) \approx f_{\theta_0}(\mathbf{x}) +
\nabla_\theta f_{\theta_0}(\mathbf{x})^\top(\theta-\theta_0)$ — treating the network as *linear*
in its parameter updates, even though $f_\theta$ itself is a highly nonlinear function of
$\mathbf{x}$. This linear-in-parameters approximation is normally hopelessly crude for a
finite-width network trained for many steps. Jacot, Gabriel & Hongler's key result is that it
becomes *exact*, for the entire duration of training, in the infinite-width limit.

### The Neural Tangent Kernel

Define $\Theta(\mathbf{x},\mathbf{x}') = \nabla_\theta f_\theta(\mathbf{x})^\top
\nabla_\theta f_\theta(\mathbf{x}')$ — the inner product of the network's parameter-gradients at
two inputs, exactly the Gram-matrix-of-gradients construction `08.01`'s attention discussion and
`11.02`'s kernel machinery both anticipate. As width $\to\infty$ (under the **NTK
parameterization**: weights initialized i.i.d. $\mathcal{N}(0,1)$, with the $1/\sqrt{\text{fan-in}}$
scaling applied explicitly in the forward pass rather than folded into the initialization
variance), two remarkable things happen, both verified numerically below: $\Theta$ converges to
a fixed, deterministic kernel independent of the random initialization, and $\Theta$ stays
**constant throughout training** — it does not depend on how far training has progressed.

### The lazy-training regime

When $\Theta$ stays fixed during training, the network's evolution reduces to *kernel gradient
descent* in function space with the fixed kernel $\Theta$: exactly the representer-theorem
machinery from `11.02`, with $\Theta$ playing the role $k$ played there. This is called the
**lazy-training regime** (or "kernel regime"): parameters do move during training, but the
*function they compute* moves as if the network were a fixed, linear-in-a-huge-feature-space
model — no genuine internal representation learning happens, in a precise sense. This is
verified below by directly comparing the closed-form NTK-kernel-regression prediction against
what a real, trained wide network actually outputs.

### Kernel regime vs. feature-learning regime

Finite (and especially narrow) networks behave very differently: $\Theta$ visibly changes during
training, verified below by a large, direct measurement of the relative change in $\Theta$
before vs. after training a narrow network, contrasted against a wide one under identical
conditions. This regime is where **feature learning** actually happens — the network's internal
representations reorganize in response to data, rather than the training dynamics being
equivalent to a fixed kernel from start to finish. Which regime a real, finite-width network
behaves closer to (and whether that's even desirable) remains an active research question; this
notebook's job is just to make the mathematical distinction concrete and measurable, not to
settle which regime describes practice best.

---

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Computing the empirical NTK $\Theta(x,x')$ for the same fixed pair of points, across many
independent random initializations and increasing width, shows the values concentrating onto a
single deterministic curve — exactly the "converges to a fixed kernel" claim, made visible.

In [ ]:
class NTKLinear(nn.Module):
    # NTK parameterization: i.i.d. N(0,1) weights, with 1/sqrt(fan_in) scaling applied
    # EXPLICITLY in the forward pass (not folded into the init variance). This is what makes
    # the NTK converge to a fixed limit as width grows, instead of blowing up or vanishing.
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_in = d_in
        self.weight = nn.Parameter(torch.randn(d_out, d_in))
        self.bias = nn.Parameter(torch.randn(d_out))
    def forward(self, x):
        return x @ self.weight.T / np.sqrt(self.d_in) + self.bias

def make_ntk_mlp(width, depth=2, seed=0):
    torch.manual_seed(seed)
    layers = []
    d_in = 1
    for _ in range(depth):
        layers += [NTKLinear(d_in, width), nn.ReLU()]
        d_in = width
    layers += [NTKLinear(d_in, 1)]
    return nn.Sequential(*layers)

def compute_ntk_value(model, x, x_prime):
    params = list(model.parameters())
    out_x = model(x.unsqueeze(0)).squeeze()
    grads_x = torch.autograd.grad(out_x, params, create_graph=False, retain_graph=True)
    grads_x_flat = torch.cat([g.flatten() for g in grads_x])
    out_xp = model(x_prime.unsqueeze(0)).squeeze()
    grads_xp = torch.autograd.grad(out_xp, params, create_graph=False, retain_graph=True)
    grads_xp_flat = torch.cat([g.flatten() for g in grads_xp])
    return (grads_x_flat @ grads_xp_flat).item()

x_probe_viz = torch.tensor([0.5])
x_other_viz = torch.tensor([1.0])

widths_viz = [5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000]
n_seeds = 6
all_vals = {w: [compute_ntk_value(make_ntk_mlp(w, seed=s), x_probe_viz, x_other_viz) for s in range(1, n_seeds + 1)]
            for w in widths_viz}

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for w in widths_viz:
    vals = all_vals[w]
    ax.scatter([w] * len(vals), vals, color="#4C72B0", alpha=0.5, s=30)
means = [np.mean(all_vals[w]) for w in widths_viz]
ax.plot(widths_viz, means, color="#C44E52", lw=2, marker='o', label="mean across seeds")
ax.set_xscale('log')
ax.set_xlabel("network width (log scale)"); ax.set_ylabel("$\\Theta(x=0.5,\\, x'=1.0)$")
ax.set_title("The empirical NTK concentrates onto a single deterministic\nvalue as width grows, independent of the random initialization")
ax.legend()
plt.show()

stds = [np.std(all_vals[w]) for w in widths_viz]
print("Standard deviation of Theta(0.5, 1.0) across", n_seeds, "random seeds, by width:")
for w, s in zip(widths_viz, stds):
    print(f"  width={w:5d}: std={s:.4f}")
print(f"\nVariance shrinks by {stds[0]/stds[-1]:.1f}x from width={widths_viz[0]} to width={widths_viz[-1]},")
print(f"confirming concentration toward a single deterministic limiting kernel.")

## 🐍 Implementation from Scratch

We compute the empirical NTK directly from per-sample parameter gradients, confirm it converges
to a fixed value as width grows, measure how much it actually changes during training for narrow
vs. wide networks (the lazy-training claim, tested directly rather than assumed), and check
whether NTK-based kernel regression predicts a genuinely trained wide network's output.

In [ ]:
class NTKLinear(nn.Module):
    # NTK parameterization: i.i.d. N(0,1) weights, 1/sqrt(fan_in) scaling applied EXPLICITLY
    # in the forward pass (not folded into the init variance) -- this is what keeps the NTK
    # convergent to a fixed limit as width grows, rather than blowing up or vanishing.
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_in = d_in
        self.weight = nn.Parameter(torch.randn(d_out, d_in))
        self.bias = nn.Parameter(torch.randn(d_out))
    def forward(self, x):
        return x @ self.weight.T / np.sqrt(self.d_in) + self.bias

def make_ntk_mlp(width, depth=2, seed=0):
    torch.manual_seed(seed)
    layers = []
    d_in = 1
    for _ in range(depth):
        layers += [NTKLinear(d_in, width), nn.ReLU()]
        d_in = width
    layers += [NTKLinear(d_in, 1)]
    return nn.Sequential(*layers)

def compute_ntk_row(model, x, X_ref):
    # Theta(x, x_i) = grad_theta f(x) . grad_theta f(x_i), summed over ALL parameters
    params = list(model.parameters())
    out_x = model(x.unsqueeze(0)).squeeze()
    grads_x = torch.autograd.grad(out_x, params, create_graph=False, retain_graph=True)
    grads_x_flat = torch.cat([g.flatten() for g in grads_x])
    vals = []
    for i in range(X_ref.shape[0]):
        out_xi = model(X_ref[i:i+1]).squeeze()
        grads_xi = torch.autograd.grad(out_xi, params, create_graph=False, retain_graph=True)
        grads_xi_flat = torch.cat([g.flatten() for g in grads_xi])
        vals.append((grads_x_flat @ grads_xi_flat).item())
    return np.array(vals)

def compute_ntk_matrix(model, X1, X2):
    params = list(model.parameters())
    def all_grads(X):
        out = []
        for i in range(X.shape[0]):
            o = model(X[i:i+1]).squeeze()
            g = torch.autograd.grad(o, params, create_graph=False, retain_graph=True)
            out.append(torch.cat([gg.flatten() for gg in g]))
        return torch.stack(out)
    G1 = all_grads(X1)
    G2 = G1 if X2 is X1 else all_grads(X2)
    return (G1 @ G2.T).detach().numpy()

# --- 1) The empirical NTK converges to a FIXED value as width grows, at several point pairs at once ---
x_probe = torch.tensor([0.5])
X_ref = torch.tensor([[0.5], [0.3], [-0.7], [1.2]])
print("Empirical NTK Theta(0.5, X_ref) at initialization, for increasing width:")
for width in [10, 100, 1000, 5000]:
    model = make_ntk_mlp(width, seed=1)
    print(f"  width={width:5d}: {np.round(compute_ntk_row(model, x_probe, X_ref), 4)}")
print("Values stabilize by width~1000 and barely move from 1000 to 5000 -- convergence, not drift.")

# --- 2) Lazy-training regime: does Theta actually stay fixed DURING training? Narrow vs wide. ---
def target_fn(x):
    return torch.sin(3 * x)

X_train = torch.linspace(-2, 2, 20).reshape(-1, 1)
y_train = target_fn(X_train)

print("\nRelative change in Theta(0.5, X_ref) from before to after training (lower = more 'lazy'):")
for width, lr in [(20, 0.2), (2000, 0.2)]:
    model = make_ntk_mlp(width, seed=1)
    ntk_before = compute_ntk_row(model, x_probe, X_ref)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for step in range(200):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(X_train), y_train)
        loss.backward()
        opt.step()
    ntk_after = compute_ntk_row(model, x_probe, X_ref)
    rel_change = np.linalg.norm(ntk_after - ntk_before) / np.linalg.norm(ntk_before)
    print(f"  width={width:5d}: final loss={loss.item():.5f}, relative Theta change={rel_change:.4f}")
print("The narrow network's kernel moves far more during training -- genuine feature learning.")
print("The wide network's kernel barely moves -- the 'lazy training' / kernel regime.")

# --- 3) Does NTK-based kernel regression predict what a REAL trained wide network outputs? ---
X_train_small = torch.linspace(-2, 2, 8).reshape(-1, 1)
y_train_small = target_fn(X_train_small)
X_test = torch.linspace(-2, 2, 12).reshape(-1, 1)

print("\nNTK-predicted vs. actually-trained network output, at increasing width:")
for width, lr, n_steps in [(300, 0.15, 800), (3000, 0.15, 800)]:
    model = make_ntk_mlp(width, seed=1)
    with torch.no_grad():
        f_train_init = model(X_train_small).squeeze().numpy()
        f_test_init = model(X_test).squeeze().numpy()
    Theta_tt = compute_ntk_matrix(model, X_train_small, X_train_small)
    Theta_te_tr = compute_ntk_matrix(model, X_test, X_train_small)

    # Linearized-model theory's prediction: f_test = f_test(init) + Theta(test,train) Theta(train,train)^-1 (y - f(init))
    alpha = np.linalg.solve(Theta_tt + 1e-6 * np.eye(len(X_train_small)), y_train_small.squeeze().numpy() - f_train_init)
    ntk_prediction = f_test_init + Theta_te_tr @ alpha

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for step in range(n_steps):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(X_train_small), y_train_small)
        loss.backward()
        opt.step()
    with torch.no_grad():
        actual_prediction = model(X_test).squeeze().numpy()

    rel_err = np.linalg.norm(ntk_prediction - actual_prediction) / np.linalg.norm(actual_prediction)
    print(f"  width={width:5d}: final train loss={loss.item():.5f}, "
          f"NTK-prediction vs actual relative error={rel_err:.4f}")
print("The gap between the NTK's PREDICTED output and the network's ACTUAL trained output")
print("shrinks as width grows -- direct evidence that wide networks really do train like")
print("kernel regression with this fixed kernel, not just approximately in some vague sense.")

## 🤖 Why This Matters for AI

The NTK theory gives one of the field's leading explanations for a genuinely puzzling
observation: heavily overparameterized networks (far more parameters than training examples,
easily capable of fitting the training data in infinitely many different ways) reliably
generalize well anyway, rather than overfitting to noise the way classical statistical learning
theory would predict. The NTK framework's answer: gradient descent from typical random
initialization doesn't converge to just *any* interpolating function — in the lazy-training
regime, it converges to the *specific* interpolator with minimum norm with respect to the NTK's
own RKHS (recall `11.02`'s representer theorem, now with $\Theta$ as the kernel). This is an
**implicit regularization** effect: no explicit penalty term is added anywhere, yet the training
dynamics themselves prefer the "simplest" (NTK-norm-smallest) function among all those that fit
the data. Below, we confirm this concretely: given only 6 noisy training points, a naive
polynomial interpolator (which also fits the training data almost exactly) and an NTK-implied
minimum-norm interpolator both achieve near-zero training error, but the NTK-implied one
generalizes substantially better across the whole input domain — direct evidence for why
"heavily overparameterized" doesn't have to mean "overfit."

In [ ]:
def target_fn_np(x):
    return np.sin(3 * x)

# Deliberately FEW training points (n=6), so a degree-5 polynomial has just enough freedom to
# interpolate them exactly -- an intentionally extreme "overparameterized-vs-data" setup.
n_train = 6
X_train_np = np.array([-1.8, -1.0, -0.3, 0.3, 1.0, 1.8])
y_train_np = target_fn_np(X_train_np) + np.random.default_rng(0).normal(scale=0.05, size=n_train)
X_test_np = np.linspace(-2, 2, 200)
y_test_true = target_fn_np(X_test_np)

# Naive interpolator: an exact-fit degree-(n-1) polynomial through every training point.
# This is a LEGITIMATE way to hit zero training error, with no regularization of any kind.
poly_coeffs = np.polyfit(X_train_np, y_train_np, deg=n_train - 1)
poly_pred = np.polyval(poly_coeffs, X_test_np)

# NTK-implied interpolator: the function a wide network's gradient descent actually converges
# to when trained to (near) zero loss -- the MIN-NORM interpolator w.r.t. the NTK's RKHS norm.
model = make_ntk_mlp(width=3000, seed=1)
X_train_t = torch.tensor(X_train_np, dtype=torch.float32).reshape(-1, 1)
X_test_t = torch.tensor(X_test_np, dtype=torch.float32).reshape(-1, 1)
with torch.no_grad():
    f_train_init = model(X_train_t).squeeze().numpy()
    f_test_init = model(X_test_t).squeeze().numpy()

Theta_tt = compute_ntk_matrix(model, X_train_t, X_train_t)
Theta_te_tr = compute_ntk_matrix(model, X_test_t, X_train_t)
reg = 1e-8  # near-zero: genuine interpolation, not ridge-style smoothing
alpha = np.linalg.solve(Theta_tt + reg * np.eye(n_train), y_train_np - f_train_init)
ntk_pred = f_test_init + Theta_te_tr @ alpha

ntk_train_fit = f_train_init + Theta_tt @ alpha
print(f"Both interpolators fit the {n_train} training points almost exactly:")
print(f"  Polynomial max training error: {np.max(np.abs(np.polyval(poly_coeffs, X_train_np) - y_train_np)):.2e}")
print(f"  NTK-implied max training error: {np.max(np.abs(ntk_train_fit - y_train_np)):.2e}")

test_err_poly = np.sqrt(np.mean((poly_pred - y_test_true)**2))
test_err_ntk = np.sqrt(np.mean((ntk_pred - y_test_true)**2))
print(f"\nTest RMSE over the WHOLE domain (not just the training points):")
print(f"  Naive polynomial interpolator:        {test_err_poly:.4f}")
print(f"  NTK-implied (min-norm) interpolator:  {test_err_ntk:.4f}")
print(f"  NTK-implied interpolator generalizes better: {test_err_ntk < test_err_poly}")

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(X_test_np, y_test_true, color='k', lw=2, ls=':', label='true target function')
ax.plot(X_test_np, poly_pred, color="#C44E52", lw=2, label=f'polynomial interpolator (test RMSE={test_err_poly:.3f})')
ax.plot(X_test_np, ntk_pred, color="#4C72B0", lw=2, label=f'NTK-implied interpolator (test RMSE={test_err_ntk:.3f})')
ax.scatter(X_train_np, y_train_np, color='k', s=60, zorder=5, label='training points (both fit these exactly)')
ax.set_ylim(-2.5, 2.5)
ax.set_title("Two interpolators, both exact on training data, generalize\nvery differently -- gradient descent implicitly prefers the better one")
ax.legend(fontsize=8.5, loc='lower center')
plt.show()

## 🏋️ Exercises

### Warm-up
1. In the Implementation section's first check, look at how `Theta(0.5, X_ref)`'s values change
   from `width=10` to `width=100` versus from `width=1000` to `width=5000`. Which jump is
   larger? Explain why this matches "convergence" (differences should shrink, not stay
   constant, as width grows) rather than the kernel just wandering randomly.

### Practice
2. In the lazy-vs-feature-learning comparison, add a third setting: `width=200` (between the
   two already tested). Report its relative $\Theta$ change and confirm it falls between the
   `width=20` and `width=2000` values — consistent with "laziness" being a continuous spectrum
   controlled by width, not a sharp on/off switch.

### Challenge
3. In the AI section, increase `n_train` from 6 to 15 (add more training points spanning
   `[-2, 2]`, keeping the same noise level) and re-run both the polynomial and NTK-implied
   interpolator comparison (you'll need a correspondingly higher-degree polynomial, still degree
   `n_train - 1`, to keep it an exact interpolator). Does the naive polynomial's generalization
   gap versus the NTK-implied interpolator shrink or grow with more data? Relate your answer to
   what happens to a high-degree interpolating polynomial's behavior *between* data points as
   more points are packed into the same fixed interval (Runge's phenomenon).

---

## 📚 Further Reading
- Jacot, Gabriel & Hongler, [\"Neural Tangent Kernel: Convergence and Generalization in Neural Networks\"](https://arxiv.org/abs/1806.07572)
- Lee et al., [\"Wide Neural Networks of Any Depth Evolve as Linear Models Under Gradient Descent\"](https://arxiv.org/abs/1902.06720)
- Belkin, Ma & Mandal, [\"To Understand Deep Learning We Need to Understand Kernel Learning\"](https://arxiv.org/abs/1802.01396)

---

*Next notebook: [Infinite-Width Networks](04_infinite_width_networks.ipynb)*